In [ ]:
import csv
import json
import sys
from pathlib import Path

import pandas as pd
from pymatgen.core import Structure
from tqdm.auto import tqdm

In [ ]:
ROOT = Path("../data")

In [ ]:
def load_dataset(path: str | Path) -> dict[int, list[str]]:
    """
    Args:
        path: Path to the dataset root.

    Returns:
        dict: Spacegroup number to list of structures in that spacegroup.
    """
    if not isinstance(path, Path):
        path = Path(path)

    json_files = sorted(path.rglob("*.json"))
    csv_files = sorted(path.rglob("*.csv"))
    files = json_files + csv_files

    try:
        files.remove(path / "CSG.csv")
    except ValueError:
        pass

    dataset = {i: [] for i in range(1, 231)}

    for datafile in tqdm(files):
        if datafile.suffix == ".json":
            data = json.loads(datafile.read_text())
            idx = int(datafile.stem.split("_")[-1])
            dataset[idx] += [entry["structure"] for entry in data]

        elif datafile.suffix == ".csv":
            data = pd.read_csv(datafile)
            file_root = datafile.parent

            for idx, group in tqdm(
                data.groupby("Space Group Number"), desc=f"Loading {datafile.name}"
            ):
                filtered_data = [file_root / "by_id" / f"{x}.CIF" for x in group["MaterialId"]]
                structures = [
                    Structure.from_str(data.read_text(), fmt="cif").to(fmt="poscar")
                    for data in tqdm(filtered_data, desc="Files in chunk", leave=False)
                ]
                dataset[idx].extend(structures)

    return dataset

In [ ]:
def filter_dataset(dataset: dict[int, list[str]]) -> dict[int, list[str]]:
    """
    Filter the dataset to only include unique compositions for each spacegroup.

    Args:
        dataset: Spacegroup number to list of structures in that spacegroup.

    Returns:
        dict: Spacegroup number to list of structures in that spacegroup.
    """
    filtered_dataset = {}
    for i in tqdm(range(1, 231), desc="Filtering dataset chunks"):
        filtered_dataset[i] = []
        seen_formulas = set()
        for struct_str in dataset[i]:
            struct = Structure.from_str(struct_str, fmt="poscar")

            formula = struct.composition.formula
            if formula not in seen_formulas:
                seen_formulas.add(formula)
                filtered_dataset[i].append(struct_str)
    return filtered_dataset

In [ ]:
dataset = load_dataset(ROOT)
dataset_length = sum(len(v) for v in dataset.values())

In [ ]:
filtered_dataset = filter_dataset(dataset)

In [ ]:
def sizeof_fmt(num, suffix="B"):
    for unit in ("", "Ki", "Mi", "Gi", "Ti", "Pi", "Ei", "Zi"):
        if abs(num) < 1024.0:
            return f"{num:3.1f}{unit}{suffix}"
        num /= 1024.0
    return f"{num:.1f}Yi{suffix}"

In [ ]:
filter_dataset_length = sum(len(v) for v in filtered_dataset.values())

print(f"Original dataset length: {dataset_length}")
print(f"Filtered dataset length: {filter_dataset_length}")
print(
    f"Dataset length reduction: {dataset_length - filter_dataset_length} ({(dataset_length - filter_dataset_length) / dataset_length * 100:.2f}%)",
    end="\n\n",
)

dataset_size = sum(sys.getsizeof(v) for v in dataset.values())
filtered_dataset_size = sum(sys.getsizeof(v) for v in filtered_dataset.values())

print(f"Memory usage of the original dataset: {sizeof_fmt(dataset_size)}")
print(f"Memory usage of the filtered dataset: {sizeof_fmt(filtered_dataset_size)}")
print(
    f"Memory reduction: {sizeof_fmt(dataset_size - filtered_dataset_size)} ({(dataset_size - filtered_dataset_size) / dataset_size * 100:.2f}%)"
)

In [ ]:
for i in range(1, 231):
    if len(dataset[i]) != len(filtered_dataset[i]):
        print(f"Spacegroup {i}: {len(dataset[i])} -> {len(filtered_dataset[i])}")

In [ ]:
pbar = tqdm(total=filter_dataset_length, desc="Writing to CSV")
with open(ROOT / "CSG.csv", "w") as csv_file:
    writer = csv.writer(csv_file)
    for key, values in filtered_dataset.items():
        for val in values:
            writer.writerow([key, val])
            pbar.update(1)

In [ ]:
df = pd.read_csv(ROOT / "csg" / "raw" / "CSG.csv")

In [ ]:
df.rename(columns={"0": "SpaceGroupNumber", "1": "Structure"}, inplace=True)

In [ ]:
df.to_csv(ROOT / "csg" / "raw" / "CSG.csv", index=False)

In [ ]:
import hashlib


def md5(fname):
    hash_md5 = hashlib.md5(usedforsecurity=False)
    with open(fname, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()


print(f"MD5 hash of the dataset: {md5(ROOT / 'CSG.csv')}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
sg_count = np.array([len(filtered_dataset.get(k, [])) for k in filtered_dataset.keys()])

In [ ]:
available_sg = sg_count[sg_count > 0]
masked_count = sg_count[sg_count > 100]
imbalance = np.max(masked_count) / np.min(masked_count)

print(f"Number of space groups: {len(available_sg)}")
print(f"Number of relevant space group: {len(masked_count)}")
print(f"Imbalance: {imbalance}")

In [ ]:
print(f"Unavailable space groups: {np.where(sg_count == 0)[0]+1}")

In [ ]:
plt.bar(np.arange(1, 231), sg_count, log=True, width=1)

plt.xlabel("Space group")
plt.ylabel("Number of entries")
plt.xlim(1, 230)